In [3]:
# Instalação e configuração no Google Colab
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium beautifulsoup4 > /dev/null

# Imports
import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Configuração do navegador
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Variáveis
links = []
pagina = 1

# Coleta de todos os links
while True:
    url = f"https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page={pagina}"
    print(f"🔎 Página {pagina}: {url}")
    driver.get(url)
    time.sleep(3)

    elementos = driver.find_elements(By.CSS_SELECTOR, "div.vtex-search-result-3-x-galleryItem a")
    if not elementos:
        print("🚫 Nenhum produto encontrado nesta página. Encerrando.")
        break

    novos_links = 0
    for i in range(len(elementos)):
        try:
            produto = driver.find_elements(By.CSS_SELECTOR, "div.vtex-search-result-3-x-galleryItem a")[i]
            href = produto.get_attribute("href")
            if href and href not in links:
                links.append(href)
                novos_links += 1
        except:
            continue

    print(f"➡️ Total acumulado: {len(links)}")
    if novos_links == 0:
        print("⚠️ Nenhum novo link encontrado nesta página. Encerrando.")
        break

    pagina += 1
    time.sleep(2)

driver.quit()

# Pasta para salvar imagens
os.makedirs("imagens_telhanorte", exist_ok=True)

# Função para salvar imagem
def salvar_imagem(imagem_url, nome_arquivo):
    try:
        response = requests.get(imagem_url, stream=True)
        if response.status_code == 200:
            nome_arquivo = re.sub(r'[\\/*?:"<>|]', "", nome_arquivo)
            caminho = os.path.join("imagens_telhanorte", f"{nome_arquivo}.jpg")
            with open(caminho, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            print(f"🖼️ Imagem salva: {caminho}")
        else:
            print(f"❌ Imagem não encontrada: {imagem_url}")
    except Exception as e:
        print(f"⚠️ Erro ao baixar imagem: {e}")

# Função para extrair detalhes
def extrair_detalhes_produto(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None
        soup = BeautifulSoup(response.text, "html.parser")

        nome = soup.find("h1", class_="vtex-store-components-3-x-productNameContainer")
        nome = nome.text.strip() if nome else "N/A"

        codigo = soup.find("span", class_="vtex-product-identifier-0-x-product-identifier__value")
        codigo = codigo.text.strip() if codigo else "0"

        preco_original = soup.select_one(".vtex-product-price-1-x-listPriceWithUnitMultiplier strike")
        preco_original = preco_original.text.strip() if preco_original else "N/A"

        preco_promocional = soup.select_one(".telhanorte-telha-store-app-1-x-preco-best-un")
        preco_promocional = preco_promocional.text.strip() if preco_promocional else "N/A"

        preco_m2 = soup.select_one(".telhanorte-telha-store-app-1-x-preco-m2")
        preco_m2 = preco_m2.text.strip() if preco_m2 else "N/A"

        imagem = soup.find("img", class_="telhanorte-telha-store-app-1-x-imagem-destaque")
        imagem = imagem['src'] if imagem else "N/A"

        return [nome, codigo, preco_original, preco_promocional, preco_m2, imagem, url]
    except Exception as e:
        print(f"⚠️ Erro ao processar {url}: {e}")
        return None

# Coleta de dados
dados = []
for i, link in enumerate(links):
    print(f"[{i+1}/{len(links)}] Coletando: {link}")
    resultado = extrair_detalhes_produto(link)
    if resultado:
        dados.append(resultado)

        codigo_produto = resultado[1].strip()
        codigo_produto = re.sub(r'[\\/*?:"<>|]', "", codigo_produto)

        if codigo_produto and codigo_produto != "0":
            salvar_imagem(resultado[5], codigo_produto)
        else:
            nome_fallback = re.sub(r'[\\/*?:"<>|]', "", resultado[0])[:50]
            salvar_imagem(resultado[5], nome_fallback)

# Salvar CSV com data
data_hoje = datetime.now().strftime("%Y-%m-%d")
arquivo_csv = f"telhanorte_porcelanatos_{data_hoje}.csv"

df = pd.DataFrame(dados, columns=[
    "Nome", "Código", "Preço Original", "Preço Promocional", "Preço por m²", "Imagem", "URL"
])
df.to_csv(arquivo_csv, index=False)
print(f"\n✅ Dados salvos em: {arquivo_csv}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔎 Página 1: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=1
➡️ Total acumulado: 40
🔎 Página 2: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=2
➡️ Total acumulado: 80
🔎 Página 3: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=3
➡️ Total acumulado: 120
🔎 Página 4: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=4
➡️ Total acumulado: 160
🔎 Página 5: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=5
➡️ Total acumulado: 200
🔎 Página 6: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos?page=6
➡️ Total acumulado: 240
🔎 Página 7: https://www.telhanorte.com.br/pisos-e-revestimentos/porcelan